In [1]:
# Run this in a JupyterLab notebook inside the Docker environment
import sys
print(f"Python: {sys.version}")
print(f"Running inside: {sys.prefix}")
print("Course environment is ready!")

Python: 3.11.10 | packaged by conda-forge | (main, Oct 16 2024, 01:19:04) [GCC 13.3.0]
Running inside: /opt/conda
Course environment is ready!


In [6]:
%%file producer.py

from kafka import KafkaProducer

import json, random, time

from datetime import datetime
 
producer = KafkaProducer(

    bootstrap_servers='broker:9092',

    value_serializer=lambda v: json.dumps(v).encode('utf-8')

)
 
shops = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']

cathegories = ['elektronika', 'odzież', 'żywność', 'książki']
 
def generate_transaction():

    return {

        'tx_id': f'TX{random.randint(1000,9999)}',

        'user_id': f'u{random.randint(1,20):02d}',

        'amount': round(random.uniform(5.0, 5000.0), 2),

        'store': random.choice(shops),

        'category': random.choice(cathegories),

        'timestamp': datetime.now().isoformat(),

    }
 
 
for i in range(1000):

    tx = generate_transaction()

    producer.send('transactions', value=tx)

    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")

    time.sleep(1)
 
producer.flush()

producer.close()

Overwriting producer.py


In [3]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

# YOUR CODE
# For each message:
#   1. Increment store_counts[store]
#   2. Add amount to total_amount[store]
#   3. Every 10 messages, print a summary table:
#      Store | Count | Total Amount | Avg Amount

Writing consumer_count.py


In [7]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Listening for large transactions (amount > 1000)...")

for message in consumer:
    tx = message.value
    if tx.get('amount', 0) > 1000:
        print(f"ALERT: {tx.get('tx_id')} | {tx.get('amount')}| {tx.get('store')} | {tx.get('category')}")

Overwriting consumer_filter.py


In [8]:
%%file consumer_velocity.py
from kafka import KafkaConsumer
import json
from datetime import datetime, timedelta

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='velocity-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_history = {}

print("Velocity Detection...")

for message in consumer:
    tx = message.value
    user_id = tx.get('user_id')
    
    current_time = datetime.fromisoformat(tx.get('timestamp'))
    
    if user_id not in user_history:
        user_history[user_id] = []
    
    user_history[user_id].append(current_time)
    
    sixty_seconds_ago = current_time - timedelta(seconds=60)
    user_history[user_id] = [t for t in user_history[user_id] if t > sixty_seconds_ago]
    
    if len(user_history[user_id]) > 3:
        print(f"VELOCITY ALERT: User {user_id} made {len(user_history[user_id])} transactions within 60 seconds!")
        print(f"   More details: {tx['tx_id']} | {tx['amount']} PLN | {tx['store']}")
        print("-" * 50)

Writing consumer_velocity.py
